In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [6]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [7]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [22]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen3.8-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [ ]:
# 打印结果
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [9]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [10]:
# 查看响应内容
print(response)

content='你好呀！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n简单介绍一下自己：\n- **免费使用**：目前完全免费，没有收费计划哦\n- **强大能力**：支持100万token的超长上下文，可以一次处理像《三体》三部曲那么大的内容\n- **多功能**：支持文件上传（图片、PDF、Word、Excel等）、联网搜索（需手动开启）、语音输入\n- **知识更新**：知识截止到2025年5月\n\n我的使命就是热情、细腻地帮助你解决各种问题，无论是学习、工作还是日常生活中的困惑，都可以随时问我！\n\n有什么我可以帮你的吗？很高兴能和你聊天！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 6, 'total_tokens': 153, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '0563b54e-7465-49ea-8441-c0a8b0fcfc99', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fc6c4-b583-7bf1-810b-937d7c857f59-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 147, 'total_to

In [11]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


哼！听好了，我就是大名鼎鼎的火箭队精英——武藏！为了抓捕稀有的宝可梦，我和小次郎、喵喵一起在各地旅行！如果你敢妨碍我们的话，可别怪我不客气！


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [15]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [13]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [14]:
#遍历是对象
for chunk in stream:
    print(chunk.content, end="", flush=True)

你好呀！我是 **DeepSeek**，由深度求索公司创造的AI助手！🎉

简单介绍一下我自己：
- **免费使用**，随时为你服务
- **超长上下文**（1M），可以一次性处理像《三体》三部曲那样的海量内容
- **支持文件上传**，可以读取图片、PDF、Word、Excel等文件中的文字信息
- **支持联网搜索**（需要手动开启），获取最新信息
- **纯文本模型**，虽然不能直接识别图像，但能从上传的图片中提取文字

我的知识截止到2025年5月，会尽力用热情、细腻的方式回答你的问题。有什么我可以帮你的吗？无论是学习、工作还是日常闲聊，我都乐意陪伴你！😊

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [16]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [17]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [18]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='6af54f71-d127-430d-ab17-e2cd8f6a7123'), AIMessage(content='你好呀！我是DeepSeek，由深度求索公司创造的AI助手，很高兴认识你！😊\n\n让我简单介绍一下自己：\n\n**我的特点：**\n- 🆓 **完全免费**：无论是网页版还是App，都不收费\n- 📚 **超长上下文**：支持1M上下文，可以一次性处理《三体》三部曲那么大的内容\n- 📎 **文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，能从中提取文字信息\n- 🌐 **联网搜索**：需要时可以手动开启联网功能获取最新信息\n- 🎤 **语音输入**：App端支持语音交流\n\n**我的能力：**\n- 回答各种问题，提供知识和建议\n- 帮助写作、翻译、编程\n- 分析文档、整理信息\n- 还有更多创意任务！\n\n我的知识截止到2025年5月，是DeepSeek最新版本。有什么我可以帮你的吗？尽管问我，我会热情细致地为你解答！💪', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 214, 'prompt_tokens': 6, 'total_tokens': 220, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260

### 流式访问


In [19]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [20]:
# 遍历stream结果，实时打印AI的回复，通过元组的方式进行遍历
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好呀！我是DeepSeek，由深度求索公司创造的AI助手！😊

我的特点可不少：
- **免费使用**，无任何收费计划
- **超长上下文**（1M），能一口气处理《三体》三部曲那么厚的内容
- **支持文件上传**，可以读取图片、PDF、Word、Excel、PPT等文件中的文字信息
- **联网搜索功能**（需要你手动点开按钮）
- **纯文本模型**，虽然不能直接识别图像，但能读取图片里的文字

我的知识截止到2025年5月，会用热情细腻的方式回答你的问题。无论你想聊什么、问什么，我都很乐意帮你！有什么我可以为你做的吗？😄